# Moving barcode sequences to ORF read description

To couple the ORF sequences with their corresponding barcode sequences, we will move the barcode sequences to the `BC:Z:` field of the description of the ORF reads. 

This way, we can easily link the ORF sequences with their corresponding barcodes for downstream analysis.

In [ ]:
%%bash

# First unzip the FASTQ files to ensure we can read it properly as an index
pigz -p 12 -k -d -c /media/niek/4TB_SSD2/analyses/GPS_ONT/orf_30.fastq.gz > /media/niek/4TB_SSD2/analyses/GPS_ONT/orf_30.fastq
pigz -p 12 -k -d -c /media/niek/4TB_SSD2/analyses/GPS_ONT/orf_24.fastq.gz > /media/niek/4TB_SSD2/analyses/GPS_ONT/orf_24.fastq

# Then put them together in a single file
cat /media/niek/4TB_SSD2/analyses/GPS_ONT/orf_24.fastq /media/niek/4TB_SSD2/analyses/GPS_ONT/orf_30.fastq > /media/niek/4TB_SSD2/analyses/GPS_ONT/orfs.fastq
rm /media/niek/4TB_SSD2/analyses/GPS_ONT/orf_24.fastq
rm /media/niek/4TB_SSD2/analyses/GPS_ONT/orf_30.fastq

# Also put the barcode txt files together
cat /media/niek/4TB_SSD2/analyses/GPS_ONT/barcode_24.txt /media/niek/4TB_SSD2/analyses/GPS_ONT/barcode_30.txt > /media/niek/4TB_SSD2/analyses/GPS_ONT/barcodes.txt

In [ ]:
#!/usr/bin/env python3
import gzip
import logging
import os
import tqdm
from Bio import SeqIO


# Reset logging handlers to allow re-configuration in Jupyter
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

# Set up logging as we cannot print everything to the console
logging.basicConfig(
    filename="move_barcode_to_description.log",
    level=logging.INFO,
    format="%(asctime)s:%(levelname)s: %(message)s",
)

input_fastq = "/media/niek/4TB_SSD2/analyses/GPS_ONT/orfs.fastq"
input_text = "/media/niek/4TB_SSD2/analyses/GPS_ONT/barcodes.txt"
output_fastq = "/media/niek/4TB_SSD2/analyses/GPS_ONT/orfs_barcode_description.fastq.gz"

# Open the text file line by line and store headers and 
# sequences in a dictionary for quick access
logging.info(f"Reading barcode information from {input_text}...")
with open(input_text, "r") as txt_file:
    header_dict = {}
    for line in txt_file:
        parts = line.strip().split(maxsplit=1)
        if len(parts) < 2:
            continue
        seq = parts[0]
        full_header = parts[1]
        uuid = full_header.split()[0]  # Extract UUID for matching
        header_dict[uuid] = seq

total_orfs = 0
orfs_with_barcodes = 0

# Interate through the barcode dictionary and update the FASTQ records accordingly
logging.info(f"Starting merge for {len(header_dict)} entries...")

total_processed = 0
matches_found = 0

with gzip.open(output_fastq, "wt") as out_file:
    # Use parse instead of index_db to avoid SQLite unique constraints
    for record in tqdm.tqdm(SeqIO.parse(input_fastq, "fastq")):
        total_processed += 1
        uuid = record.id
        
        if uuid in header_dict:
            # Append the barcode to the header
            barcode = header_dict[uuid]
            record.id = uuid
            # BC: tag name (arbitrary)
            # Z: tag type (printable string)
            # This format allows us to easily extract the 
            # barcode later using e.g. pysam.
            record.description = f"{uuid} BC:Z:{barcode}"
            
            SeqIO.write(record, out_file, "fastq")
            matches_found += 1
        
        # for testing, only process the first 1000 records
        #if total_processed == 10000:
        #    break

logging.info(f"Done. Processed {total_processed} records. Found {matches_found} matches.")

798144it [10:49, 1220.84it/s]

In [ ]:
%%bash

# Show first record of the FASTQ file to confirm it looks correct
zcat "/media/niek/4TB_SSD2/analyses/GPS_ONT/orfs_barcode_description.fastq.gz" | head -n 4

@00a2870a-79fb-4e1e-a961-34da5bd0a6fd BC:Z:TAATAGGCTGGGAGTGGTCAGTCC
ATGGAATATCATCCTGATTTAGAAAATTTGGATGAAGATGGATATACTCAATTACACTTCGACTCTCAAAGCAATACCAGGATAGCTGTTGTTTCAGAGAAAGGATTGTGTGCTGCATCTCCTCCTTGGCGCCTCATTGCTGTAATTTTGGGAATCCTATGCTTGGTAATACTGGTGATAGCTGTGGTCCTGGGTACCATGGGGGTTCTTTCCAGCCCTTGTCCTCCTAATTGGATTATATATGAGAAGAGCTGTTATCTATTCAGCATGTCACTAAATTCCTGGGATGGAAGTAAAAGACAATGCTGGCAACTGGGCTCTAATCTCCTAAAGATAGACAGCTCAAATGAATTGGTAAGTGTAGACTTCTGTTATGATTATCTGTGGTGTGTATCCTAG
+
PLGKGJHHGFCDDDESHKIOSRPFFEEGFHGSISIKJPKNKFFIDEFFIJMHGJGHIJGHJIKJIMHKPGGGGGEEFFGFFEDFEGGGHLPRONIKSMGNHH(((((BFGCCEDSLHJHKILHHHGHGIIPSSPQNJJOISRKROLIFBJHKNMMMLLSNINKIJNMKQLHSJIKKNKJNJJNLMMKRPGFHICGNSLSMSNHJJSPQNOQIMMKOSMSPLJGKKKSGGFHHJGJGLHPILKQQNNSSJJQMKSLSOKJKIJGHMJKSLLNOLPPNLNKJEHDHIJJNNPSJSOSSNJNNKMFGF88876@DDCGGIHGGJGMSSJLLHSMSPOLOLPIHNOKKJJQNSMIKKJIQIOIILHIKOLJLISKKQHIIPNMOISOSQNLPSLSLIHIMOSN
